In [1]:
import os
import pathlib
import sys
import time

import pandas as pd
import psutil
import tomli
from image_analysis_3D.featurization_utils.feature_writing_utils import (
    format_morphology_feature_name,
    save_features_as_parquet,
)
from image_analysis_3D.featurization_utils.intensity_utils import (
    measure_3D_intensity_CPU,
)
from image_analysis_3D.featurization_utils.loading_classes import (
    ImageSetLoader,
    ObjectLoader,
)
from image_analysis_3D.featurization_utils.resource_profiling_util import (
    start_profiling,
    stop_profiling,
)
from image_analysis_3D.file_utils.arg_parsing_utils import (
    check_for_missing_args,
    parse_args,
)
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()
image_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot")).resolve(), root_dir
)

In [ ]:
if not in_notebook:
    arguments_dict = parse_args()
    patient = arguments_dict["patient"]
    well_fov = arguments_dict["well_fov"]
    channel = arguments_dict["channel"]
    compartment = arguments_dict["compartment"]
    processor_type = arguments_dict["processor_type"]
    input_subparent_name = arguments_dict["input_subparent_name"]
    mask_subparent_name = arguments_dict["mask_subparent_name"]
    output_features_subparent_name = arguments_dict["output_features_subparent_name"]

else:
    well_fov = "C4-2"
    patient = "NF0055_T1"
    channel = "DNA"
    compartment = "Nuclei"
    processor_type = "CPU"
    input_subparent_name = "zstack_images"
    mask_subparent_name = "segmentation_masks"
    output_features_subparent_name = "extracted_features"

image_set_path = pathlib.Path(
    f"{image_base_dir}/data/{patient}/{input_subparent_name}/{well_fov}/"
)
mask_set_path = pathlib.Path(
    f"{image_base_dir}/data/{patient}/{mask_subparent_name}/{well_fov}/"
)
output_parent_path = pathlib.Path(
    f"{root_dir}/data/{patient}/{output_features_subparent_name}/{well_fov}/"
)
output_parent_path.mkdir(parents=True, exist_ok=True)
channel_mapping_file_path = pathlib.Path(
    f"{root_dir}/config/channel_mapping.toml"
).resolve(strict=True)

In [ ]:
# read in channel mapping
with open(channel_mapping_file_path, "rb") as f:
    channel_mapping_dict = tomli.load(f)
channel_n_compartment_mapping = channel_mapping_dict["channel_mapping"]

{'DNA': '405',
 'ER': '488',
 'AGP': '555',
 'Mito': '640',
 'BF': 'TRANS',
 'Nuclei': 'nuclei_',
 'Cell': 'cell_',
 'Cytoplasm': 'cytoplasm_',
 'Organoid': 'organoid_'}

In [13]:
start_time, start_mem = start_profiling()

In [14]:
image_set_loader = ImageSetLoader(
    image_set_path=image_set_path,
    mask_set_path=mask_set_path,
    anisotropy_spacing=(1, 0.1, 0.1),
    channel_mapping=channel_n_compartment_mapping,
    image_set_name=well_fov,
    mask_key_name=[channel_n_compartment_mapping[compartment]],
    raw_image_key_name=[channel_n_compartment_mapping[channel]],
)

In [15]:
object_loader = ObjectLoader(
    image_set_loader.image_set_dict[channel],
    image_set_loader.image_set_dict[compartment],
    channel,
    compartment,
)

In [16]:
if processor_type == "CPU":
    output_dict = measure_3D_intensity_CPU(object_loader)
else:
    raise ValueError(f"Processor type {processor_type} is not supported. Use 'CPU'.")
final_df = pd.DataFrame(output_dict)
# prepend compartment and channel to column names
final_df = final_df.pivot(
    index=["object_id"],
    columns="feature_name",
    values="value",
).reset_index()
final_df.rename(
    columns={
        col: format_morphology_feature_name(
            compartment=compartment,
            channel=channel,
            feature_type="Intensity",
            measurement=col,
        )
        if col != "object_id"
        else col
        for col in final_df.columns
    },
    inplace=True,
)

final_df.insert(0, "image_set", image_set_loader.image_set_name)

save_path = save_features_as_parquet(
    parent_path=output_parent_path,
    df=final_df,
    feature_type="Intensity",
    channel=channel,
    compartment=compartment,
    cpu_or_gpu=processor_type,
)
final_df.head()

feature_name,image_set,object_id,Organoid_DNA_Intensity_CMI-X,Organoid_DNA_Intensity_CMI-Y,Organoid_DNA_Intensity_CMI-Z,Organoid_DNA_Intensity_IntegratedIntensity,Organoid_DNA_Intensity_IntegratedIntensityEdge,Organoid_DNA_Intensity_LowerQuartileIntensity,Organoid_DNA_Intensity_MassDisplacement,Organoid_DNA_Intensity_MaxIntensity,...,Organoid_DNA_Intensity_MaxZ,Organoid_DNA_Intensity_MeanAbsoluteDeviationIntensity,Organoid_DNA_Intensity_MeanIntensity,Organoid_DNA_Intensity_MeanIntensityEdge,Organoid_DNA_Intensity_MedianIntensity,Organoid_DNA_Intensity_MinIntensity,Organoid_DNA_Intensity_MinIntensityEdge,Organoid_DNA_Intensity_StdIntensity,Organoid_DNA_Intensity_StdIntensityEdge,Organoid_DNA_Intensity_UpperQuartileIntensity
0,C4-2,1,1097.562866,413.687164,0.998956,4429652.0,489585.0,1542.0,0.121540,5911.0,...,2.0,313.843414,1834.224487,886.929321,1799.0,771.0,0.0,466.398712,951.264832,2056.0
1,C4-2,2,1204.064819,1218.938354,1.499412,4369514.0,430218.0,1542.0,0.123533,5911.0,...,2.0,329.790894,1856.208130,911.478821,1799.0,771.0,0.0,479.410828,992.327698,2056.0


In [17]:
stop_profiling(
    start_time=start_time,
    start_mem=start_mem,
    feature_type="Intensity",
    well_fov=well_fov,
    patient_id=patient,
    channel=channel,
    compartment=compartment,
    CPU_GPU=processor_type,
    output_file_dir=pathlib.Path(
        f"{root_dir}/data/{patient}/extracted_features/run_stats/{well_fov}_{channel}_{compartment}_Intensity_{processor_type}.parquet"
    ),
)


        Memory and time profiling for the run:
        Patient ID: NF0055_T1
        Well and FOV: C4-2
        Feature type: Intensity
        CPU/GPU: CPU
        Peak memory (tracemalloc): 4445.14 MB
        Current memory (tracemalloc): 941.13 MB
        RSS at end: 1140.92 MB
        Time elapsed:
        --- 29.86 seconds ---
        --- 0.50 minutes ---
        --- 0.01 hours ---
    


True